# Этап 2b — Two-Tower training (Colab)

Запускать в Google Colab (обучение NN, локально не тянется).

Код — через git, данные — через Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/recsys_pet_project/data"
OUT_DIR = f"{DRIVE_DIR}/two_tower_out"

import os
os.makedirs(OUT_DIR, exist_ok=True)
print(os.listdir(DRIVE_DIR))

## Код проекта — git clone/pull (репозиторий публичный)

In [ ]:
REPO_DIR = "/content/ML_practice"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/vadim-white/ML_practice.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/recsys-pet-project")

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.id_mapping import build_id_maps
from src.two_tower.model import TwoTowerModel
from src.two_tower.dataset import TwoTowerDataset
from src.negative_sampling import NEG_RATIO

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| NEG_RATIO:", NEG_RATIO)

## Данные

In [ ]:
train = pd.read_parquet(f"{DRIVE_DIR}/train.parquet")
train_positive = train[train["is_positive"] == 1]

user_map, item_map = build_id_maps(train)
print(f"users: {len(user_map)}, items: {len(item_map)}, positive interactions: {len(train_positive)}")

## Обучение

In [ ]:
EMBEDDING_DIM = 64
BATCH_SIZE = 2048
EPOCHS = 40
LR = 1e-3

dataset = TwoTowerDataset(train_positive, user_map, item_map, neg_ratio=NEG_RATIO)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)

# logQ correction (Yi et al. 2019): без неё модель занижает популярные товары,
# т.к. они непропорционально часто встречаются как негативы (in-batch и explicit).
item_log_prob = torch.tensor(dataset.sampler.log_probs(), dtype=torch.float32, device=device)

model = TwoTowerModel(
    num_users=len(user_map),
    num_items=len(item_map),
    embedding_dim=EMBEDDING_DIM,
    item_log_prob=item_log_prob,
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

for epoch in range(EPOCHS):
    total_loss = 0.0
    for user_ids, pos_item_ids, neg_item_ids in loader:
        user_ids, pos_item_ids, neg_item_ids = user_ids.to(device), pos_item_ids.to(device), neg_item_ids.to(device)
        optimizer.zero_grad()
        loss = model(user_ids, pos_item_ids, neg_item_ids)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(user_ids)
    print(f"epoch {epoch+1}/{EPOCHS}: loss={total_loss / len(dataset):.4f}")

## Экспорт эмбеддингов на Drive

Сохраняем эмбеддинги для **всех** id из id-map (не только встреченных в
батчах — embedding-таблица покрывает всех), плюс сам маппинг idx→raw id,
чтобы локальный `02c` мог сопоставить строки эмбеддингов с исходными
MovieLens ID.

In [ ]:
model.eval()
all_user_idx = torch.arange(len(user_map), device=device)
all_item_idx = torch.arange(len(item_map), device=device)

user_embeddings = model.user_embeddings(all_user_idx).cpu().numpy()
item_embeddings = model.item_embeddings(all_item_idx).cpu().numpy()

np.save(f"{OUT_DIR}/user_embeddings.npy", user_embeddings)
np.save(f"{OUT_DIR}/item_embeddings.npy", item_embeddings)

import json
with open(f"{OUT_DIR}/idx2user.json", "w") as f:
    json.dump({int(k): int(v) for k, v in user_map.idx_to_raw.items()}, f)
with open(f"{OUT_DIR}/idx2item.json", "w") as f:
    json.dump({int(k): int(v) for k, v in item_map.idx_to_raw.items()}, f)

print("saved:", os.listdir(OUT_DIR))